### 구글 드라이브 마운트 & 파일 경로 입력

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
seismic_path = "/content/drive/MyDrive/TriAI/그로쓰/Data/processed/seismic/hokkaido/hokkaido_seismic_360_180.npz"

### 변형 전 데이터셋 구조

In [ ]:
import numpy as np

seismic = np.load(seismic_path)

data = seismic['data']
station_names = seismic['station_names']
start_times = seismic['start_times']

print(f"data: {data.shape}, station_names: {station_names}, start_times: {start_times}")

Exception ignored in: <function NpzFile.__del__ at 0x786f963c5bc0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 226, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/_npyio_impl.py", line 221, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected


data: (1092, 36000, 3), station_names: ['N.AGWH' 'N.AGWH' 'N.AGWH' ... 'N.YUBH' 'N.YUBH' 'N.YUBH'], start_times: ['2018-09-06T02:51:00.000000Z' '2018-09-06T02:54:00.000000Z'
 '2018-09-06T02:57:00.000000Z' ... '2018-09-06T03:24:00.000000Z'
 '2018-09-06T03:27:00.000000Z' '2018-09-06T03:30:00.000000Z']


### 데이터셋 구조 변형 & 확인

In [ ]:
NUM_WINDOWS = 14  # 관측소당 윈도우 수
OUTPUT_PATH = '/content/drive/MyDrive/TriAI/그로쓰/Data/processed/seismic/hokkaido/hokkaido_seismic_360_180_ver2.npz'

# 8개씩 묶었을 때 같은 관측소명인지 확인
total_windows, time_len, channels = data.shape
assert total_windows % NUM_WINDOWS == 0, \
    f"전체 윈도우 수({total_windows})가 {NUM_WINDOWS}의 배수가 아닙니다!"

num_stations = total_windows // NUM_WINDOWS

# 각 그룹의 station_name이 모두 동일한지 체크
for i in range(num_stations):
    group_names = station_names[i * NUM_WINDOWS : (i + 1) * NUM_WINDOWS]
    assert len(set(group_names)) == 1, \
        f"관측소 {i}번 그룹의 station_names가 일치하지 않습니다: {group_names}"

print("모든 그룹의 관측소명이 일치합니다.")

# 구조 변형
new_data = data.reshape(num_stations, NUM_WINDOWS, time_len, channels)
new_station_names = station_names[::NUM_WINDOWS]
new_start_times = start_times.reshape(num_stations, NUM_WINDOWS)

# 변형 확인
print(f"data: {new_data.shape}, station_names: {new_station_names.shape}, start_times: {new_start_times.shape}")

모든 그룹의 관측소명이 일치합니다.
data: (78, 14, 36000, 3), station_names: (78,), start_times: (78, 14)


### 변형 데이터셋 저장 & 확인

In [ ]:
# 저장
np.savez_compressed(
    OUTPUT_PATH,
    data = new_data,
    station_names = new_station_names,
    start_times = new_start_times)

# 저장 확인
verify = np.load(OUTPUT_PATH, allow_pickle=True)
print(f"data: {verify['data'].shape}, station_names: {verify['station_names'].shape}, start_times: {verify['start_times'].shape}")